# Data Merge — Combining YouTube, Hacker News & GitHub Datasets
**COSC 3047 Assignment 2**

This notebook standardises and combines comment data from three platforms into a single unified dataset for NLP and network analysis.

**Sources:**
- **YouTube** (23,871 comments) — video comments and replies on AI coding assistant reviews
- **Hacker News** (~2,200 comments) — technical discussion threads on AI coding tools
- **GitHub** (~5,800 comments) — issue discussions

**Pipeline:**
1. Load and standardise all three datasets to a common schema
2. Combine into `combined_comments.csv` for NLP analysis
3. Build combined edge list for network analysis:
   - YouTube: directed reply edges (from existing `reply_edges.json`)
   - HN: directed reply edges (derived from `parentId` → `commentId`)
   - HN + GitHub: bipartite user→thread edges (from teammate-provided edge files)

**Output:** `data/processed/combined/`

## 1. Setup & Imports

In [ ]:
import json
import html
import re
from pathlib import Path
from itertools import combinations

import pandas as pd
import numpy as np

In [ ]:
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

ALT_DATA_ROOT = PROJECT_ROOT.parent / "data"
LOCAL_DATA_ROOT = PROJECT_ROOT / "data"

OUTPUT_DIR = LOCAL_DATA_ROOT / "processed/combined"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def first_existing(*paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return Path(paths[0])


# YouTube data may sit in assign2/data while this notebook lives in social-media-2.
YT_COMMENTS_PATH = first_existing(
    LOCAL_DATA_ROOT / "processed/youtube/YTCommentsCleaned.csv",
    ALT_DATA_ROOT / "processed/youtube/YTCommentsCleaned.csv",
)
YT_REPLY_JSON_PATH = first_existing(
    LOCAL_DATA_ROOT / "processed/youtube/reply_edges.json",
    ALT_DATA_ROOT / "processed/youtube/reply_edges.json",
)
YT_COMMENTER_VIDEO_EDGES_PATH = first_existing(
    LOCAL_DATA_ROOT / "processed/youtube/YTCommenterVideoEdges.csv",
    ALT_DATA_ROOT / "processed/youtube/YTCommenterVideoEdges.csv",
)

# Hacker News
HN_COMMENTS_PATH = first_existing(
    LOCAL_DATA_ROOT / "processed/hackernews/hackernews_comments.csv",
    ALT_DATA_ROOT / "processed/hackernews/hackernews_comments.csv",
)
HN_CLEANED_PATH = first_existing(
    LOCAL_DATA_ROOT / "processed/hackernews/hackernews_text_cleaned.csv",
    ALT_DATA_ROOT / "processed/hackernews/hackernews_text_cleaned.csv",
)
HN_EDGES_PATH = first_existing(
    LOCAL_DATA_ROOT / "processed/hackernews/hackernews_user_story_edges.csv",
    ALT_DATA_ROOT / "processed/hackernews/hackernews_user_story_edges.csv",
)

# GitHub
GH_COMMENTS_PATH = first_existing(
    LOCAL_DATA_ROOT / "processed/github/GitHubDiscussionTextFiltered.csv",
    ALT_DATA_ROOT / "processed/github/GitHubDiscussionTextFiltered.csv",
)
GH_EDGES_PATH = first_existing(
    LOCAL_DATA_ROOT / "processed/github/GitHubUserIssueEdges.csv",
    ALT_DATA_ROOT / "processed/github/GitHubUserIssueEdges.csv",
)

print(f"Project root:      {PROJECT_ROOT}")
print(f"Local data root:   {LOCAL_DATA_ROOT}")
print(f"Alt data root:     {ALT_DATA_ROOT}")
print(f"Output directory:  {OUTPUT_DIR}")
print()
for label, path in [
    ("YT comments",      YT_COMMENTS_PATH),
    ("YT reply JSON",    YT_REPLY_JSON_PATH),
    ("YT edge CSV",      YT_COMMENTER_VIDEO_EDGES_PATH),
    ("HN comments",      HN_COMMENTS_PATH),
    ("HN cleaned",       HN_CLEANED_PATH),
    ("HN edges",         HN_EDGES_PATH),
    ("GH comments",      GH_COMMENTS_PATH),
    ("GH edges",         GH_EDGES_PATH),
]:
    status = "Done" if path.exists() else "Missing"
    print(f"  {status:7s} {label:15s} {path}")


## 2. Standard Schema

All datasets are mapped to this unified schema before combining:

| Column | Description |
|---|---|
| `platform` | youtube / hackernews / github |
| `author` | Username |
| `text` | Raw comment text |
| `text_clean` | Cleaned text for NLP |
| `published_at` | Timestamp |
| `parent_author` | Parent commenter (reply edges) |
| `comment_type` | top_level or reply |
| `source_id` | Video ID / Story ID / Issue key |
| `source_title` | Video / Story / Issue title |
| `like_count` | Likes (0 if unavailable) |
| `comment_id` | Unique ID within platform |

## 3. YouTube

In [ ]:
yt_raw = pd.read_csv(YT_COMMENTS_PATH)

comment_type = yt_raw["commentType"] if "commentType" in yt_raw.columns else pd.Series("top_level", index=yt_raw.index)
parent_author = yt_raw["parentAuthor"] if "parentAuthor" in yt_raw.columns else pd.Series(None, index=yt_raw.index)
comment_like_count = yt_raw["commentLikeCount"] if "commentLikeCount" in yt_raw.columns else pd.Series(0, index=yt_raw.index)

# If the cleaned file does not contain replies, treat all rows as top-level comments.
yt = pd.DataFrame({
    "platform":      "youtube",
    "author":        yt_raw["commentAuthor"],
    "text":          yt_raw["commentText"],
    "text_clean":    yt_raw["commentTextClean"],
    "published_at":  yt_raw["commentPublishedAt"],
    "parent_author": parent_author,
    "comment_type":  comment_type,
    "source_id":     yt_raw["videoId"],
    "source_title":  yt_raw["title"],
    "like_count":    comment_like_count.fillna(0).astype(int),
    "comment_id":    yt_raw.index.astype(str) + "_yt"
})

print(f"YouTube: {len(yt):,} comments")
print(f"  top_level: {len(yt[yt.comment_type == 'top_level']):,}")
print(f"  replies:   {len(yt[yt.comment_type == 'reply']):,}")
print(f"  authors:   {yt['author'].nunique():,}")


## 4. Hacker News

We use `hackernews_text_cleaned.csv` for pre-cleaned text and filter to `recordType == 'comment'` only — stories are excluded from the comment analysis.

Reply edges are derived from `hackernews_comments.csv` by joining `parentId` → `commentId` to resolve parent authors.

In [ ]:
# Use cleaned file — filter to comments only (exclude story records)
hn_cleaned = pd.read_csv(HN_CLEANED_PATH)
hn_cleaned = hn_cleaned[hn_cleaned["recordType"] == "comment"].copy()

# Use raw comments file to resolve parent authors for reply edges
hn_raw = pd.read_csv(HN_COMMENTS_PATH)
id_to_author = hn_raw.set_index("commentId")["commentAuthor"].to_dict()
story_ids = set(hn_raw["storyId"].unique())

def resolve_parent_author(row):
    if row["parentId"] in story_ids:
        return None
    return id_to_author.get(row["parentId"], None)

hn_raw["parent_author"] = hn_raw.apply(resolve_parent_author, axis=1)
hn_raw["comment_type"]  = hn_raw["parent_author"].apply(
    lambda x: "reply" if pd.notna(x) else "top_level"
)

# Merge cleaned text back onto raw (join on commentId → recordId)
hn_merged = hn_raw.merge(
    hn_cleaned[["recordId", "textClean"]],
    left_on="commentId",
    right_on="recordId",
    how="left"
)
hn_merged["textClean"] = hn_merged["textClean"].fillna(
    hn_merged["commentText"].fillna("")
)

hn = pd.DataFrame({
    "platform":      "hackernews",
    "author":        hn_merged["commentAuthor"],
    "text":          hn_merged["commentText"].fillna(""),
    "text_clean":    hn_merged["textClean"],
    "published_at":  hn_merged["commentCreatedAt"],
    "parent_author": hn_merged["parent_author"],
    "comment_type":  hn_merged["comment_type"],
    "source_id":     hn_merged["storyId"].astype(str),
    "source_title":  hn_merged["storyTitle"],
    "like_count":    0,
    "comment_id":    hn_merged["commentId"].astype(str) + "_hn"
})

print(f"Hacker News: {len(hn):,} comments")
print(f"  top_level: {len(hn[hn.comment_type == 'top_level']):,}")
print(f"  replies:   {len(hn[hn.comment_type == 'reply']):,}")
print(f"  authors:   {hn['author'].nunique():,}")

## 5. GitHub

We use `GitHubDiscussionTextFiltered.csv` and keep only rows where `include_for_nlp == True`.
Both `issue` and `comment` record types are kept since issue bodies are substantive text.
GitHub has no reply structure so `comment_type` is `top_level` for all.

In [ ]:
gh_raw = pd.read_csv(GH_COMMENTS_PATH)

# Keep only NLP-suitable rows
gh_filtered = gh_raw[gh_raw["include_for_nlp"] == True].copy()

gh = pd.DataFrame({
    "platform":      "github",
    "author":        gh_filtered["author"],
    "text":          gh_filtered["text"].fillna(""),
    "text_clean":    gh_filtered["textClean"].fillna(""),
    "published_at":  gh_filtered["createdAt"],
    "parent_author": None,
    "comment_type":  "top_level",
    "source_id":     gh_filtered["issueKey"],
    "source_title":  gh_filtered["issueKey"].str.split("#").str[-1],
    "like_count":    0,
    "comment_id":    gh_filtered["recordId"].astype(str) + "_gh"
})

print(f"GitHub: {len(gh):,} comments")
print(f"  issues:   {gh_raw[gh_raw.recordType == 'issue'].shape[0]:,}")
print(f"  comments: {gh_raw[gh_raw.recordType == 'comment'].shape[0]:,}")
print(f"  authors:  {gh['author'].nunique():,}")
print(f"  (excluded {len(gh_raw) - len(gh_filtered):,} rows flagged for exclusion)")

## 6. Combine All Comments

In [ ]:
combined = pd.concat([yt, hn, gh], ignore_index=True)

# Remove empty text
combined = combined[combined["text_clean"].str.strip().str.len() > 0].copy()
combined = combined.reset_index(drop=True)

combined.to_csv(OUTPUT_DIR / "combined_comments.csv", index=False)

print(f"Combined dataset: {len(combined):,} comments")
print()
print("By platform:")
print(combined["platform"].value_counts().to_string())
print()
print("By comment type:")
print(combined["comment_type"].value_counts().to_string())
print()
print(f"Unique authors: {combined['author'].nunique():,}")
print(f"Saved combined_comments.csv")

## 7. Build Combined Edge List

Three edge types are combined:

| Platform | Edge type | Direction | Source |
|---|---|---|---|
| YouTube | reply | directed | existing `reply_edges.json` |
| HN | reply | directed | derived from `parentId` resolution |
| HN | user→story | bipartite | `hackernews_user_story_edges.csv` |
| GitHub | user→issue | bipartite | `GitHubUserIssueEdges.csv` |

All edges are tagged with `platform` and `edge_type` so the network notebook can filter by type.

In [ ]:
all_edges = []

# 1. YouTube reply edges. Prefer direct reply JSON if present; otherwise fall back to
# commenter-video participation edges so YouTube remains available for community detection.
if YT_REPLY_JSON_PATH.exists():
    with open(YT_REPLY_JSON_PATH, encoding="utf-8") as f:
        yt_edges = json.load(f)
    for e in yt_edges:
        all_edges.append({
            "source":    e["source"],
            "target":    e["target"],
            "weight":    int(e.get("weight", 1)),
            "platform":  "youtube",
            "edge_type": "reply"
        })
    print(f"YouTube reply edges:      {len(yt_edges):,}")
elif YT_COMMENTER_VIDEO_EDGES_PATH.exists():
    yt_bipartite = pd.read_csv(YT_COMMENTER_VIDEO_EDGES_PATH)
    for _, row in yt_bipartite.iterrows():
        all_edges.append({
            "source":    row["source"],
            "target":    row["target"],
            "weight":    int(row.get("weight", 1)),
            "platform":  "youtube",
            "edge_type": "user_thread"
        })
    print(f"YouTube reply edges:      0")
    print(f"YouTube user→video edges: {len(yt_bipartite):,}")
else:
    print("YouTube edges:            0")

# 2. HN reply edges derived from parentId resolution.
hn_replies = hn[hn["comment_type"] == "reply"][["author", "parent_author"]].dropna()
hn_reply_agg = (
    hn_replies
    .groupby(["author", "parent_author"])
    .size()
    .reset_index(name="weight")
)
for _, row in hn_reply_agg.iterrows():
    all_edges.append({
        "source":    row["author"],
        "target":    row["parent_author"],
        "weight":    int(row["weight"]),
        "platform":  "hackernews",
        "edge_type": "reply"
    })
print(f"HN reply edges:           {len(hn_reply_agg):,}")

# 3. HN user→story bipartite edges.
hn_bipartite = pd.read_csv(HN_EDGES_PATH)
for _, row in hn_bipartite.iterrows():
    all_edges.append({
        "source":    row["source"],
        "target":    row["target"],
        "weight":    int(row["weight"]),
        "platform":  "hackernews",
        "edge_type": "user_thread"
    })
print(f"HN user→story edges:      {len(hn_bipartite):,}")

# 4. GitHub user→issue bipartite edges.
gh_bipartite = pd.read_csv(GH_EDGES_PATH)
for _, row in gh_bipartite.iterrows():
    all_edges.append({
        "source":    row["source"],
        "target":    row["target"],
        "weight":    int(row["weight"]),
        "platform":  "github",
        "edge_type": "user_thread"
    })
print(f"GitHub user→issue edges:  {len(gh_bipartite):,}")

with open(OUTPUT_DIR / "combined_edges.json", "w", encoding="utf-8") as f:
    json.dump(all_edges, f, ensure_ascii=False, indent=2)

print(f"\nTotal edges: {len(all_edges):,}")
print("Saved combined_edges.json")


## 8. Summary

In [ ]:
edges_df = pd.DataFrame(all_edges)

print("=" * 55)
print("Data Merge Complete")
print("=" * 55)
print(f"  Total comments:       {len(combined):,}")
print(f"    YouTube:            {len(yt):,}")
print(f"    Hacker News:        {len(hn):,}")
print(f"    GitHub:             {len(gh):,}")
print()
print(f"  Total edges:          {len(all_edges):,}")
print(edges_df.groupby(["platform", "edge_type"]).size().to_string())
print()
print(f"  Unique authors:       {combined['author'].nunique():,}")
print()
print("Outputs saved to data/processed/combined/:")
for f in sorted(OUTPUT_DIR.glob("*")):
    size = f.stat().st_size / 1024
    print(f"  {f.name} ({size:.0f} KB)")
print("=" * 55)
print()

## 9. Outputs

| File | Description |
|---|---|
| `combined_comments.csv` | All comments in unified schema — input for `text_analysis.ipynb` |
| `combined_edges.json` | All edges with `platform` and `edge_type` tags — input for `network_analysis.ipynb` |

### Edge types in combined_edges.json
- `reply` — directed (YouTube + HN): use for PageRank and directed centrality
- `user_thread` — bipartite (HN + GitHub): use for community detection and co-participation analysis

The network notebook filters by `edge_type` depending on the analysis being performed.